In [1]:
import os
import sys

llava_path = "/home/cbn-gpu12/FNF/VLM/LLaVA/LLaVA"
sys.path.append(llava_path)
from llava.eval.run_llava import eval_model

[2025-03-18 17:04:51,177] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [2]:
import os
os.chdir('/home/cbn-gpu12/FNF/VLM/LLaVA/LLaVA-Med')


from peft import PeftModel
from huggingface_hub import create_repo

import sys
import warnings
warnings.filterwarnings("ignore")
import random
import torch
from torch.utils.data.dataset import Dataset
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import io
import requests
from datetime import datetime
import gc
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
import json
import time 
from collections import defaultdict 
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, LoraModel, get_peft_model, prepare_model_for_kbit_training
from torch.optim.lr_scheduler import ReduceLROnPlateau 
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.conversation import Conversation
from llava.mm_utils import tokenizer_image_token, process_images
from llava.model.builder import load_pretrained_model
from llava.conversation import conv_templates
from tqdm import tqdm
from torch.utils.data import DataLoader
import glob
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from multiprocessing import Pool, cpu_count
import pydicom
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import uuid
import pickle
from transformers import LlamaTokenizer
from llava.model import LlavaMistralForCausalLM  # LLaVA-Med의 모델 클래스 import
from dotenv import load_dotenv
import wandb
from functools import lru_cache
from torchvision.transforms import Resize, Compose, ToTensor
from functools import partial  # collate_fn에 인자 전달을 위한 패키지
from torch.utils.data._utils.pin_memory import pin_memory
from transformers import AutoTokenizer, AutoModelForCausalLM
from llava.utils import disable_torch_init
from accelerate import init_empty_weights
from accelerate import Accelerator, DeepSpeedPlugin
from transformers import BitsAndBytesConfig
import cv2  # OpenCV를 활용한 빠른 이미지 저장
from huggingface_hub import notebook_login
from accelerate.utils import set_module_tensor_to_device 
from transformers.integrations import WandbCallback
from transformers.models.mistral.modeling_mistral import MistralRotaryEmbedding
import shutil
from transformers.integrations.deepspeed import HfTrainerDeepSpeedConfig
from langgraph.graph import END, StateGraph
from langchain.cache import InMemoryCache
from langchain.globals import set_llm_cache 
from typing import TypedDict, List, Dict, Any, Annotated
import operator
from llava.conversation import conv_templates, SeparatorStyle
from transformers import StoppingCriteria
from llava.utils import disable_torch_init
from enum import auto, Enum
from contextlib import redirect_stdout
load_dotenv()
set_llm_cache(InMemoryCache())    
load_dotenv()
os.environ["WANDB_API_KEY"] = ""
os.environ["HUGGING_FACE_HUB_TOKEN"] = ""
notebook_login()
wandb.login()

# CUDA 환경 변수 설정 - 메모리 초과 문제 방지
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# 사용 가능한 GPU 설정
device_count = torch.cuda.device_count()
if device_count > 1:
    print(f"🖥 {device_count}개 GPU 사용 중: {list(range(device_count))}")
    
# 캐시 디렉토리 설정  
CACHE_DIR = "/home/cbn-gpu12/FNF/VLM/LLavA/dataset/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

# 장치 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")



wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sarahyo941 (sarahyo941-university-of-ulsan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


🖥 4개 GPU 사용 중: [0, 1, 2, 3]


In [3]:
class KeywordsStoppingCriteria(StoppingCriteria):
    def __init__(self, keywords, tokenizer, input_ids_len):
        self.keywords = keywords
        self.tokenizer = tokenizer
        self.input_ids_len = input_ids_len
        self.keyword_ids = [tokenizer(keyword).input_ids for keyword in keywords]
        
    def __call__(self, output_ids, scores, **kwargs):
        if len(output_ids[0]) <= self.input_ids_len:
            return False
            
        for keyword_id in self.keyword_ids:
            if len(keyword_id) == 0:
                continue
            if output_ids[0][-len(keyword_id):].tolist() == keyword_id:
                return True
        return False

In [4]:
class LLaVAMedPipelineState(TypedDict):
    source_dir: str
    target_dir: str
    model_path: str
    output_dir: str
    train_ratio: float
    seed: int

    train_dir: Annotated[str, operator.add]
    test_dir: Annotated[str, operator.add]
    processed_items: Annotated[int, operator.add]
    
    # 데이터셋 준비 결과
    tokenizer: Any
    model: Any
    collate_fn: Any
    vqa_rad_dataset_train: Any
    vqa_rad_dataset_test: Any
    context_len: int
    
    # 학습 결과
    training_completed: bool
    lora_save_path: Annotated[str, operator.add]
    merged_save_path: Annotated[str, operator.add]



In [5]:
class DataProcessor:
    def __init__(self):
        # 미리 모델 로드 - 한 번만 로드하여 재사용
        print("LLaVA-Med 모델 초기화 중...")
        self.tokenizer, self.model, self.image_processor, _ = self._load_pretrained_model("microsoft/llava-med-v1.5-mistral-7b")
        self.model.eval()  # 추론 모드로 설정

    # 이미지 경로 변경 부분만 수정
    def _get_dcm_path(self, metadata, metadata_file):
        """메타데이터에서 DICOM 파일 경로 생성"""
        serial = metadata.get('serial', '')
        
        # 타입 체크 및 변환
        if isinstance(serial, int):
            # 정수인 경우 문자열로 변환
            folder_name = str(serial)
        else:
            # 문자열인 경우 '_' 기준으로 분리 (증강된 데이터 처리)
            folder_name = str(serial).split('_')[0]
        
        # DICOM 파일 기본 경로
        base_path = '/mnt/nas_backup/고효진/FNF/FNF_origin/FNF/Final_data/datasets/raw_internal'
        
        # 파일 타입 결정 (메타데이터 파일명 기반)
        if 'LAT' in os.path.basename(metadata_file):
            view_type = 'LAT'
        else:
            view_type = metadata.get('LR', '')
        
        # AP 뷰는 a000.dcm, 측면 뷰는 t000.dcm 사용
        if view_type in ['L', 'R']:  # AP 뷰
            dcm_filename = f"{folder_name}a000.dcm"
        else:  # Lateral 뷰
            dcm_filename = f"{folder_name}t000.dcm"
        
        # 최종 DICOM 파일 경로
        dcm_path = os.path.join(base_path, folder_name, dcm_filename)
        
        # 디버깅을 위한 출력
        print(f"이미지 파일 경로: {dcm_path}")
        
        return dcm_path
        
    def process(self, state: LLaVAMedPipelineState) -> LLaVAMedPipelineState:
        """데이터 처리 및 VQARAD 형식으로 저장 - 모델 기반 답변 생성"""
        print("===== 이미지 분석 및 VQARAD 형식으로 변환 =====")
        
        # 경로 설정
        source_dir = state["source_dir"]
        target_dir = state["target_dir"]
        source_images_dir = os.path.join(source_dir, "images")
        source_metadata_dir = os.path.join(source_dir, "metadata")
        
        # 대상 디렉토리 생성
        train_dir = os.path.join(target_dir, "train")
        test_dir = os.path.join(target_dir, "test")
        train_images_dir = os.path.join(train_dir, "images")
        test_images_dir = os.path.join(test_dir, "images")
        
        os.makedirs(train_dir, exist_ok=True)
        os.makedirs(test_dir, exist_ok=True)
        os.makedirs(train_images_dir, exist_ok=True)
        os.makedirs(test_images_dir, exist_ok=True)
        
        # 메타데이터 파일 목록 가져오기
        metadata_files = glob.glob(os.path.join(source_metadata_dir, "*.json"))
        print(f"총 {len(metadata_files)}개의 메타데이터 파일 발견")
        
        # 각 메타데이터 파일의 정보를 담을 리스트
        all_items = []
        # Garden 유형별 응답 템플릿
        garden_type_info ="""
        - Garden Type I: The key features are: - Incomplete fracture with valgus impaction, - Minimal or no cortical disruption, - Generally stable configuration. The fracture line may be subtle, often appearing as trabecular impaction rather than a clear break.
        - Garden Type II:  The characteristic features include: - Complete fracture without displacement, - Minimal disruption of trabecular pattern, - No significant angulation or rotation. Despite the complete fracture, the bone fragments remain properly aligned, making it a stable fracture.
        - Garden Type III: The diagnostic features include: - Complete fracture with partial displacement, - Some disruption of trabecular alignment, - Cortical contact partially maintained but with angulation. There may be early signs of femoral head malalignment, increasing the risk of instability and avascular necrosis."
        - Garden Type IV: The distinctive features include: - Complete fracture with full displacement, - No cortical contact between fragments, - Severe disruption of trabecular and anatomical alignment. The femoral head is completely separated from the shaft, significantly increasing the risk of avascular necrosis.
        - Normal: No significant abnormalities observed, with bone structure and alignment within normal limits.
        Analyze the type of femoral neck fracture (Garden classification) shown in this X-ray image and determine which Garden type it belongs to."""
        
        batch_size = 10  # 모델 추론에는 더 작은 배치 크기 사용

        for i in range(0, len(metadata_files), batch_size):
            batch_files = metadata_files[i:i+batch_size]
            batch_items = []
            
            for metadata_file in tqdm(batch_files):
                try:
                    with open(metadata_file, 'r', encoding='utf-8') as f:
                        metadata = json.load(f)
                        
                    # DICOM 파일 경로 가져오기
                    full_image_path = self._get_dcm_path(metadata, metadata_file)
                    
                    # 이미지 파일이 존재하는지 확인
                    if not os.path.exists(full_image_path):
                        print(f"이미지 파일이 존재하지 않음: {full_image_path}")
                        continue
                        
                    answer_label = metadata.get('label')
                    # 표준 질문
                    question = f"""
                        [Instructions]
                        You are an AI assistant specialized in radiology topics. You are a given a Femoral neck X-ray image of Garden Type {answer_label} of Femoral Neck Fracture patient. 
                        Generate medical descriptions with a consistent style. Use the following guidelines 
                        - Degree: Explain rationale for the garden type {answer_label} shown in the given image.
                        - Landmarks: Specify areas of interest (e.g., femoral neck, femoral head, greater & lesser torchanter, femoral shaft, etc.) 
                        - Features: Describe any abnormalities observed (e.g., fracture line, osteopenia, avascular necrosis, displacement, rotation, etc) 
                        - Impression: Conclude with your clinical impression (e.g., FNF garden type diagnosis) 

                        Ensure consistency and clarity of the report.
                    """
                    
                    # 모델을 사용하여 이미지 분석 및 답변 생성
                    answer = self._generate_answer_from_image(full_image_path, question)
                    
                    # 원래 레이블 저장 (나중에 평가 목적으로)
                    
                    
                    # 메타데이터에 질문과 답변 정보 추가
                    metadata_with_qa = {
                        **metadata,  # 기존 메타데이터 유지
                        "question": question,
                        "answer": answer,
                        "timestamp": datetime.now().isoformat()
                    }
                    
                    # 아이템 정보 추가
                    batch_items.append({
                        "id": f"{metadata.get('serial')}_{metadata.get('side')}",
                        "image_file": full_image_path,
                        "question": question,
                        "answer": answer,
                        "original_label": answer_label,
                        "metadata": metadata_with_qa
                    })
                    
                except Exception as e:
                    print(f"메타데이터 파일 처리 오류 ({metadata_file}): {e}")
                    import traceback
                    traceback.print_exc()
            
            # 배치 처리 결과를 전체 목록에 추가
            all_items.extend(batch_items)
            
            # 메모리 정리
            gc.collect()
            torch.cuda.empty_cache()
        
        print(f"총 {len(all_items)}개 아이템 처리 완료")
        
        # 랜덤 분할
        random.seed(state["seed"])
        random.shuffle(all_items)
        split_idx = int(len(all_items) * state["train_ratio"])
        train_data = all_items[:split_idx]
        test_data = all_items[split_idx:]
        
        print(f"전체 데이터: {len(all_items)}, 학습 데이터: {len(train_data)}, 테스트 데이터: {len(test_data)}")
        
        # VQARAD 형식으로 변환 및 이미지 복사 (배치 처리)
        train_vqarad = []
        test_vqarad = []
        
        # 메타데이터 수집 (train/test)
        train_metadata = [item["metadata"] for item in train_data]
        test_metadata = [item["metadata"] for item in test_data]
        
        # train_metadata.json, test_metadata.json 저장
        train_metadata_path = os.path.join(train_dir, "train_metadata.json")
        test_metadata_path = os.path.join(test_dir, "test_metadata.json")
        
        # 메타데이터 파일 저장
        print(f"\n==== 메타데이터 파일 저장 중 ====")
        print(f"학습 메타데이터 경로: {train_metadata_path} ({len(train_metadata)}개 항목)")
        print(f"테스트 메타데이터 경로: {test_metadata_path} ({len(test_metadata)}개 항목)")
        
        try:
            # 학습 메타데이터 저장
            with open(train_metadata_path, 'w', encoding='utf-8') as f:
                json.dump(train_metadata, f, ensure_ascii=False, indent=2)
            print(f"학습 메타데이터 저장 완료: {train_metadata_path}")
            
            # 테스트 메타데이터 저장
            with open(test_metadata_path, 'w', encoding='utf-8') as f:
                json.dump(test_metadata, f, ensure_ascii=False, indent=2)
            print(f"테스트 메타데이터 저장 완료: {test_metadata_path}")
            
            # 저장 확인
            print("\n메타데이터 저장 상태 확인:")
            if os.path.exists(train_metadata_path):
                print(f"  학습 메타데이터 파일 존재: {train_metadata_path} (크기: {os.path.getsize(train_metadata_path) / (1024*1024):.2f} MB)")
            else:
                print(f"  학습 메타데이터 파일이 존재하지 않음: {train_metadata_path}")
            
            if os.path.exists(test_metadata_path):
                print(f"  테스트 메타데이터 파일 존재: {test_metadata_path} (크기: {os.path.getsize(test_metadata_path) / (1024*1024):.2f} MB)")
            else:
                print(f"  테스트 메타데이터 파일이 존재하지 않음: {test_metadata_path}")
        except Exception as e:
            print(f"메타데이터 저장 중 오류 발생: {e}")
            import traceback
            traceback.print_exc()
        
        # 효율적인 디렉토리 생성 - 미리 필요한 모든 디렉토리 생성
        for subfolder in ["Lateral", "Left", "Right"]:
            os.makedirs(os.path.join(train_images_dir, subfolder), exist_ok=True)
            os.makedirs(os.path.join(test_images_dir, subfolder), exist_ok=True)
        
        # 학습 데이터 처리
        print("학습 데이터 처리 중...")
        self._process_dataset_batch(train_data, source_images_dir, train_images_dir, train_vqarad, batch_size=10)
        
        # 테스트 데이터 처리
        print("테스트 데이터 처리 중...")
        self._process_dataset_batch(test_data, source_images_dir, test_images_dir, test_vqarad, batch_size=10
                                    )
        
        # 변환된 데이터 저장
        train_json_path = os.path.join(train_dir, "dataset.json")
        test_json_path = os.path.join(test_dir, "dataset.json")
        
        print(f"\n==== 최종 데이터 저장 중 ====")
        print(f"학습 데이터 경로: {train_json_path} ({len(train_vqarad)}개 아이템)")
        print(f"테스트 데이터 경로: {test_json_path} ({len(test_vqarad)}개 아이템)")
        try:
            # 저장 디렉토리 확인
            os.makedirs(os.path.dirname(train_json_path), exist_ok=True)
            os.makedirs(os.path.dirname(test_json_path), exist_ok=True)
            
            # 학습 데이터 저장
            with open(train_json_path, 'w', encoding='utf-8') as f:
                json.dump(train_vqarad, f, ensure_ascii=False, indent=2)
            print(f"학습 데이터 저장 완료: {train_json_path}")
            
            # 테스트 데이터 저장
            with open(test_json_path, 'w', encoding='utf-8') as f:
                json.dump(test_vqarad, f, ensure_ascii=False, indent=2)
            print(f"테스트 데이터 저장 완료: {test_json_path}")
            
            # 저장 확인
            print("\n저장 상태 확인:")
            if os.path.exists(train_json_path):
                print(f"  학습 파일 존재: {train_json_path} (크기: {os.path.getsize(train_json_path) / (1024*1024):.2f} MB)")
            else:
                print(f"  학습 파일이 존재하지 않음: {train_json_path}")
            
            if os.path.exists(test_json_path):
                print(f"  테스트 파일 존재: {test_json_path} (크기: {os.path.getsize(test_json_path) / (1024*1024):.2f} MB)")
            else:
                print(f"  테스트 파일이 존재하지 않음: {test_json_path}")
                
        except Exception as e:
            print(f"데이터 저장 중 오류 발생: {e}")
            import traceback
            traceback.print_exc()

        # 이중으로 저장 (안전성을 위해)
        with open(train_json_path, 'w', encoding='utf-8') as f:
            json.dump(train_vqarad, f, ensure_ascii=False, indent=2)
        
        with open(test_json_path, 'w', encoding='utf-8') as f:
            json.dump(test_vqarad, f, ensure_ascii=False, indent=2)
        
        # 평가 데이터 분석 - 원래 레이블과 모델 예측 비교
        self._analyze_model_predictions(all_items)
        
        print("\n===== VQARAD 데이터 상태 확인 =====")
        print(f"학습 데이터 수: {len(train_vqarad)}")
        if len(train_vqarad) > 0:
            print("학습 데이터 첫 번째 아이템 예시:")
            print(json.dumps(train_vqarad[0], indent=2, ensure_ascii=False))
        
        print(f"테스트 데이터 수: {len(test_vqarad)}")
        if len(test_vqarad) > 0:
            print("테스트 데이터 첫 번째 아이템 예시:")
            print(json.dumps(test_vqarad[0], indent=2, ensure_ascii=False))
        
        # JSON 파일 저장 경로 설정
        train_json_path = os.path.join(train_dir, "dataset.json")
        test_json_path = os.path.join(test_dir, "dataset.json")
        
        print("\n===== JSON 파일 저장 시작 =====")
        
        try:
            # 디렉토리 존재 확인
            os.makedirs(os.path.dirname(train_json_path), exist_ok=True)
            os.makedirs(os.path.dirname(test_json_path), exist_ok=True)
            
            # 학습 데이터 저장
            if len(train_vqarad) > 0:
                print(f"학습 데이터 저장 중... ({len(train_vqarad)}개 아이템)")
                with open(train_json_path, 'w', encoding='utf-8') as f:
                    json.dump({
                        "version": "1.0",
                        "description": "FNF Classification Dataset",
                        "date_created": datetime.now().isoformat(),
                        "data": train_vqarad
                    }, f, ensure_ascii=False, indent=2)
                print(f"학습 데이터 저장 완료: {train_json_path}")
                
                # 파일 크기 확인
                if os.path.exists(train_json_path):
                    file_size = os.path.getsize(train_json_path) / (1024 * 1024)  # MB
                    print(f"저장된 파일 크기: {file_size:.2f} MB")
            else:
                print("학습 데이터가 비어있어 저장하지 않음")
            
            # 테스트 데이터 저장
            if len(test_vqarad) > 0:
                print(f"\n테스트 데이터 저장 중... ({len(test_vqarad)}개 아이템)")
                with open(test_json_path, 'w', encoding='utf-8') as f:
                    json.dump({
                        "version": "1.0",
                        "description": "FNF Classification Dataset",
                        "date_created": datetime.now().isoformat(),
                        "data": test_vqarad
                    }, f, ensure_ascii=False, indent=2)
                print(f"테스트 데이터 저장 완료: {test_json_path}")
                
                # 파일 크기 확인
                if os.path.exists(test_json_path):
                    file_size = os.path.getsize(test_json_path) / (1024 * 1024)  # MB
                    print(f"저장된 파일 크기: {file_size:.2f} MB")
            else:
                print("테스트 데이터가 비어있어 저장하지 않음")
            
            # 저장 검증
            print("\n===== 저장된 파일 검증 =====")
            if os.path.exists(train_json_path):
                with open(train_json_path, 'r', encoding='utf-8') as f:
                    saved_train_data = json.load(f)
                    print(f"학습 데이터 검증: {len(saved_train_data['data'])}개 아이템 확인됨")
            
            if os.path.exists(test_json_path):
                with open(test_json_path, 'r', encoding='utf-8') as f:
                    saved_test_data = json.load(f)
                    print(f"테스트 데이터 검증: {len(saved_test_data['data'])}개 아이템 확인됨")
                    
        except Exception as e:
            print(f"JSON 파일 저장 중 오류 발생: {e}")
            import traceback
            traceback.print_exc()
        
        print("\n===== JSON 파일 저장 완료 =====")
        
        return {
            **state,
            "train_dir": train_dir,
            "test_dir": test_dir,
            "processed_items": len(all_items)
        }

    def _generate_answer_from_image(self, image_path, question):
        os.environ["LLAVA_DEBUG"] = "1" 
        try:
            """LLaVA-Med 모델을 사용해 이미지 분석 및 답변 생성"""
            
            # DICOM 파일 검증 및 정보 출력
            if not os.path.exists(image_path):
                print(f"오류: 이미지 파일이 존재하지 않음: {image_path}")
                return "이미지 파일을 찾을 수 없습니다."
            
            # 이미지 정보 출력
            file_size = os.path.getsize(image_path) / 1024  # KB 단위
            print(f"이미지 파일 경로: {image_path}")
            print(f"이미지 파일 크기: {file_size:.2f} KB")
            
            # DICOM 파일 직접 확인 (PIL 사용하지 않고)
            if image_path.lower().endswith('.dcm'):
                try:
                    import pydicom
                    dicom_data = pydicom.dcmread(image_path)
                    print(f"DICOM 이미지 크기: {dicom_data.pixel_array.shape}")
                    print(f"DICOM 이미지 타입: {dicom_data.SOPClassUID}")
                except Exception as e:
                    print(f"DICOM 파일 읽기 오류: {e}")
            
            # 질문 정보 출력
            print(f"질문 글자 수: {len(question)}")
            print(f"질문 내용: {question[:100]}..." if len(question) > 100 else f"질문 내용: {question}")
            
            # 이후 코드는 그대로 유지 (run_llava.py의 load_image 함수가 DICOM 처리)
            # eval_model 함수에 전달할 인자 설정
            print("\n----- 모델 인자 설정 -----")
            args = type('Args', (), {
                "model_path": "microsoft/llava-med-v1.5-mistral-7b",
                "model_base": None,
                "image_file": image_path,
                "query": question,
                "conv_mode": "mistral_instruct",
                "sep": ",",
                "temperature": 0.7,  # 다양한 응답을 위해 온도 상향 조정
                "top_p": 0.9,        # top_p 값 설정 추가
                "num_beams": 1,
                "max_new_tokens": 500,
                "debug": True
            })()
                
            
            # 인자 정보 출력
            print(f"모델 경로: {args.model_path}")
            print(f"대화 모드: {args.conv_mode}")
            print(f"최대 토큰 수: {args.max_new_tokens}")
            print(f"온도(temperature): {args.temperature}")
            
            print("\n----- 모델 실행 시작 -----")
            start_time = time.time()
            
            # 출력 캡처
            f = io.StringIO()
            try:
                with redirect_stdout(f):
                    eval_model(args)
                captured_output = f.getvalue()
                print(f"캡처된 출력 길이: {len(captured_output)} 글자")
                
                # 디버깅용으로 출력 전체 표시 (이미지 처리 관련 로그를 확인하기 위함)
                print(f"캡처된 출력 전체:\n{captured_output}")
            except Exception as e:
                print(f"모델 실행 중 오류 발생: {e}")
                import traceback
                traceback.print_exc()
                return f"모델 실행 오류: {str(e)}"
            
            end_time = time.time()
            print(f"모델 실행 시간: {end_time - start_time:.2f}초")
            
            # 답변 추출 시도
            print("\n----- 답변 추출 -----")
            lines = captured_output.strip().split('\n')
            answer = ""
            
            print(f"출력 라인 수: {len(lines)}")
            # 마지막 몇 개 라인 출력
            print("마지막 10개 라인:")
            for i, line in enumerate(lines[-10:] if len(lines) >= 10 else lines):
                print(f"  라인 {len(lines) - 10 + i if len(lines) >= 10 else i}: {line[:100]}..." if len(line) > 100 else f"  라인 {len(lines) - 10 + i if len(lines) >= 10 else i}: {line}")
            
            # 답변 추출 (마지막 의미 있는 라인)
            for line in reversed(lines):
                if line.strip() and not line.startswith("Running") and not line.startswith("==="):
                    answer = line.strip()
                    break
            
            print(f"\n----- 최종 응답 -----")
            print(f"응답 길이: {len(answer)} 글자")
            print(f"응답 내용: {answer}")
            
            # 메모리 정리
            import gc
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.synchronize()
            
            print(f"===== 이미지 분석 완료: {os.path.basename(image_path)} =====\n")
            return answer  # 처리 시간도 반환
            
        except Exception as e:
            print(f"이미지 분석 처리 중 예외 발생: {e}")
            import traceback
            traceback.print_exc()
            return f"이미지 분석 중 오류가 발생했습니다: {str(e)}", 0  # 오류 시 처리 시간은 0으로 반환
            
    
    def _process_dataset_batch(self, data, source_images_dir, target_images_dir, vqarad_list, batch_size=10):
        """데이터셋 배치 처리 헬퍼 함수"""
        print(f"\n==== 배치 처리 시작 (총 {len(data)}개 아이템) ====")
        print(f"소스 이미지 디렉토리: {source_images_dir}")
        print(f"대상 이미지 디렉토리: {target_images_dir}")
        
        # 대상 디렉토리 존재 확인
        os.makedirs(target_images_dir, exist_ok=True)
        
        for i in range(0, len(data), batch_size):
            batch_items = data[i:i+batch_size]
            batch_vqarad = []
            
            print(f"\n배치 {i//batch_size + 1} 처리 중 ({len(batch_items)} 아이템)...")
            
            for idx, item in enumerate(tqdm(batch_items)):
                try:
                    image_path = item["image_file"]
                    
                    # 뷰 타입 결정 (메타데이터 기반)
                    if 'metadata' in item and 'LR' in item['metadata']:
                        view_type = item['metadata']['LR']
                        if view_type in ['L', 'R']:
                            subfolder = 'Left' if view_type == 'L' else 'Right'
                        else:
                            subfolder = 'Lateral'
                    else:
                        # 파일명으로 판단
                        if 't000.dcm' in image_path:
                            subfolder = 'Lateral'
                        else:
                            subfolder = 'AP'  # 기본값
                    
                    # 대상 디렉토리 생성
                    target_subfolder = os.path.join(target_images_dir, subfolder)
                    os.makedirs(target_subfolder, exist_ok=True)
                    
                    # 파일명 생성 (원본 파일명 유지)
                    base_filename = os.path.basename(image_path)
                    filename_without_ext = os.path.splitext(base_filename)[0]
                    target_filename = f"{filename_without_ext}.png"
                    target_path = os.path.join(target_subfolder, target_filename)
                    
                    print(f"\n처리 중인 파일:")
                    print(f"원본 경로: {image_path}")
                    print(f"대상 경로: {target_path}")
                    
                    try:
                        # DICOM 파일 읽기
                        dicom_data = pydicom.dcmread(image_path)
                        image_array = dicom_data.pixel_array
                        
                        # 정규화
                        image_array = (image_array - np.min(image_array)) / (np.max(image_array) - np.min(image_array)) * 255.0
                        image_array = image_array.astype(np.uint8)
                        
                        # PIL 이미지로 변환 및 저장
                        image = Image.fromarray(image_array).convert("RGB")
                        image.save(target_path, "PNG")
                        
                        print(f"이미지 저장 완료: {target_path}")
                        
                        # VQARAD 형식으로 변환
                        relative_path = os.path.relpath(target_path, target_images_dir)
                        vqarad_item = {
                            "id": item["id"],
                            "image": relative_path.replace("\\", "/"),  # Windows 경로 호환성
                            "conversations": [
                                {
                                    "from": "human",
                                    "value": item["question"]
                                },
                                {
                                    "from": "llava-med",
                                    "value": item["answer"]
                                }
                            ],
                            "metadata": {
                                "original_label": item.get("original_label"),
                                "view_type": subfolder
                            }
                        }
                        
                        batch_vqarad.append(vqarad_item)
                        
                    except Exception as e:
                        print(f"이미지 처리 중 오류 발생 ({image_path}): {e}")
                        continue
                    
                except Exception as e:
                    print(f"아이템 처리 중 오류 발생: {e}")
                    continue
            
            # 배치 결과 추가
            vqarad_list.extend(batch_vqarad)
            
            # 중간 결과 저장
            try:
                interim_file = os.path.join(os.path.dirname(target_images_dir), "interim_results.json")
                with open(interim_file, 'w', encoding='utf-8') as f:
                    json.dump(vqarad_list, f, ensure_ascii=False, indent=2)
                print(f"중간 결과 저장 완료: {interim_file} ({len(vqarad_list)}개 아이템)")
                
                # 디렉토리 내용 확인
                print("\n현재 저장된 파일 확인:")
                for root, dirs, files in os.walk(target_images_dir):
                    print(f"\n디렉토리: {root}")
                    for f in files:
                        file_path = os.path.join(root, f)
                        print(f"  - {f} ({os.path.getsize(file_path) / 1024:.2f} KB)")
                        
            except Exception as e:
                print(f"중간 결과 저장 중 오류 발생: {e}")
            
            # 메모리 정리
            gc.collect()
            torch.cuda.empty_cache()
        
        print(f"\n==== 배치 처리 완료 ====")
        print(f"처리된 아이템 수: {len(vqarad_list)}")        
       
    
    def _analyze_model_predictions(self, items):
        """모델 예측과 원래 레이블 비교 분석"""
        correct = 0
        total = 0
        confusion_matrix = [[0 for _ in range(4)] for _ in range(4)]
        
        for item in items:
            if "original_label" not in item or item["original_label"] is None:
                continue
            
            # 원래 레이블 (0-3)
            original_label = item["original_label"]
            
            # 모델 답변에서 Garden 유형 추출
            answer = item["answer"]
            predicted_label = None
            
            # 답변에서 Garden 유형 추출 시도
            if "Garden Type I" in answer or "Garden Type 1" in answer or "Type I" in answer:
                predicted_label = 0
            elif "Garden Type II" in answer or "Garden Type 2" in answer or "Type II" in answer:
                predicted_label = 1
            elif "Garden Type III" in answer or "Garden Type 3" in answer or "Type III" in answer:
                predicted_label = 2
            elif "Garden Type IV" in answer or "Garden Type 4" in answer or "Type IV" in answer:
                predicted_label = 3
            
            # 예측 레이블이 추출되었을 경우만 평가
            if predicted_label is not None:
                total += 1
                if predicted_label == original_label:
                    correct += 1
                
                # 혼동 행렬 업데이트
                confusion_matrix[original_label][predicted_label] += 1
        
        # 결과 출력
        if total > 0:
            accuracy = correct / total
            print(f"\n===== 모델 예측 분석 =====")
            print(f"총 평가 데이터: {total}개")
            print(f"정확도: {accuracy:.4f} ({correct}/{total})")
            
            print("\n혼동 행렬:")
            print("실제\\예측 | Type I | Type II | Type III | Type IV")
            print("-" * 50)
            for i in range(4):
                row = confusion_matrix[i]
                print(f"Type {i+1}    | {row[0]:6d} | {row[1]:6d} | {row[2]:7d} | {row[3]:6d}")
        else:
            print("평가할 데이터가 없습니다.")
    
    def _load_pretrained_model(self, model_path, device_map="auto"):
        """모델 로드 함수"""
        # GPU 메모리 정리
        torch.cuda.empty_cache()
        gc.collect()
        torch.cuda.synchronize()
        
        print(f"모델 로드 중: {model_path}")
        
        # 양자화 설정
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        
        # 메모리 설정 (GPU당 사용 가능한 메모리 설정)
        gpu_count = torch.cuda.device_count()
        max_memory = {i: "10GB" for i in range(gpu_count)}
        max_memory["cpu"] = "24GB"
        
        # 모델 로드
        model = LlavaMistralForCausalLM.from_pretrained(
            model_path,
            quantization_config=quantization_config,
            device_map=device_map,
            max_memory=max_memory,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True
        )
        
        # 토크나이저 로드
        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            use_fast=False,
            padding_side="right"
        )
        
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # 비전 타워 로드
        vision_tower = model.get_vision_tower()
        if not vision_tower.is_loaded:
            print("비전 타워 로드 중...")
            vision_tower.load_model()
            vision_tower.to(dtype=torch.bfloat16)
            print("비전 타워 로드 완료")
        
        # 패치 적용: 이미지 인코딩 메서드 동기화 (디바이스 일관성 유지)
        import types
        
        def synchronized_encode_images(self, images):
            """디바이스 동기화된 이미지 인코딩 메서드"""
            if images is None:
                return None
            
            # 필요한 모듈 가져오기
            vision_tower = self.get_model().get_vision_tower()
            mm_projector = self.get_model().mm_projector
            
            # 현재 디바이스 확인
            vision_device = next(vision_tower.parameters()).device
            mm_device = next(mm_projector.parameters()).device
            
            # 원래 이미지 디바이스 기억
            original_img_device = images.device
            
            # 1단계: vision tower와 이미지 디바이스 동기화
            if images.device != vision_device:
                images = images.to(vision_device)
            
            # 2단계: 이미지 특징 추출
            with torch.no_grad():
                image_forward_out = vision_tower.vision_tower(
                    images,
                    output_hidden_states=True
                )
                image_features = vision_tower.feature_select(image_forward_out)
            
            # 3단계: mm_projector와 이미지 특징 디바이스 동기화
            if image_features.device != mm_device:
                image_features = image_features.to(mm_device)
            
            # 4단계: 이미지 특징 프로젝션
            image_features = mm_projector(image_features)
            
            return image_features
        
        # 패치 적용
        model.encode_images = types.MethodType(synchronized_encode_images, model)
        
        # 컨텍스트 길이 계산
        if hasattr(model.config, "max_sequence_length"):
            context_len = model.config.max_sequence_length
        elif hasattr(model.config, "max_position_embeddings"):
            context_len = model.config.max_position_embeddings
        else:
            context_len = 2048
        
        return tokenizer, model, vision_tower.image_processor, context_len

In [6]:
class DatasetPreparation:
    def __init__(self):
        pass
    
    def prepare(self, state: LLaVAMedPipelineState) -> LLaVAMedPipelineState:
        """VQARAD 데이터셋 및 모델 준비"""
        print("===== 데이터셋 및 모델 준비 =====")
        
        # VQARAD 데이터셋 클래스 생성
        class VQARAD(torch.utils.data.Dataset):
            def __init__(self, root_dir, split):
                super().__init__()
                self.split = split
                self.image_folder = os.path.join(root_dir, split, "images")
                self.paths = {
                    'train': os.path.join(root_dir, 'train', 'dataset.json'),
                    'test': os.path.join(root_dir, 'test', 'dataset.json')
                }

                print(f"{split} 데이터셋 로드 중: {self.paths[self.split]}")
                with open(self.paths[self.split], 'r') as f:
                    self.dataset = json.load(f)
                
                print(f"{split} 데이터셋 크기: {len(self.dataset)}")

            def __len__(self):
                return len(self.dataset)

            def __getitem__(self, idx):
                item = self.dataset[idx]
                id = item['id']
                question = item['conversations'][0]['value']
                answer = item['conversations'][1]['value']
                image_path = item['image']
                image = Image.open(os.path.join(self.image_folder, image_path)).convert('RGB')

                return id, question, answer, image
        
        # 데이터 콜레이터 클래스
        class DataCollator:
            def __init__(self, tokenizer, split, conversation_template, pad_token_id, image_processor):
                self.tokenizer = tokenizer
                self.split = split
                self.conversation_template = conversation_template
                self.pad_token_id = pad_token_id
                self.image_processor = image_processor

            def __call__(self, rows):
                if not isinstance(rows, list):
                    rows = [rows]
                
                if self.split == "train":
                    return self._collate_train(rows)
                elif self.split == "test":
                    return self._collate_test(rows)
                else:
                    raise ValueError(f"Invalid split: {self.split}")

            def _collate_train(self, rows):
                train_input_ids_list = []
                train_labels_list = []
                train_images = []
                sample_ids = []

                for row in rows:
                    try:
                        id, question, answer, image = row
                        
                        # 이미지 처리
                        train_images.append(image)
                        
                        # 질문에서 이미지 토큰 처리
                        question = question.replace(DEFAULT_IMAGE_TOKEN, '').strip()
                        question = DEFAULT_IMAGE_TOKEN + '\n' + question

                        # 대화 형식으로 변환
                        conv = self.conversation_template.copy()
                        conv.append_message(conv.roles[0], question)
                        conv.append_message(conv.roles[1], None)
                        prefix = conv.get_prompt()

                        conv = self.conversation_template.copy()
                        conv.append_message(conv.roles[0], question)
                        conv.append_message(conv.roles[1], answer)
                        full = conv.get_prompt()

                        # 토큰화
                        prefix = self._tokenizer_image_token(prefix, self.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
                        full = self._tokenizer_image_token(full, self.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")

                        prefix_length = prefix.size(0)
                        train_input_ids = full
                        train_labels = full.clone()
                        train_labels[:prefix_length] = -100

                        train_input_ids_list.append(train_input_ids)
                        train_labels_list.append(train_labels)
                        sample_ids.append(id)

                    except Exception as e:
                        print(f"데이터 처리 오류: {e}")
                        continue

                if not train_input_ids_list:
                    raise ValueError("배치에 유효한 데이터가 없습니다")

                # 패딩 처리
                pad_value = -114514
                train_input_ids = pad_sequence(train_input_ids_list, batch_first=True, padding_value=pad_value)
                train_labels = pad_sequence(train_labels_list, batch_first=True, padding_value=pad_value)
                train_attention_mask = (train_input_ids != pad_value).long()
                
                train_input_ids[train_input_ids == pad_value] = self.pad_token_id
                train_labels[train_labels == pad_value] = -100
                
                # 이미지 처리 및 디바이스 동기화
                processed_images = self._process_images(train_images).to(torch.bfloat16)

                return {
                    "input_ids": train_input_ids,
                    "labels": train_labels,
                    "attention_mask": train_attention_mask,
                    "images": processed_images
                }
            
            def _collate_test(self, rows):
                # 기본적으로 학습과 동일한 처리
                return self._collate_train(rows)
            
            def _process_images(self, images):
                """이미지 처리 함수"""
                image_tensor = self.image_processor(images, return_tensors='pt')['pixel_values']
                return image_tensor
            
            def _tokenizer_image_token(self, prompt, tokenizer, image_token_index, return_tensors=None):
                """이미지 토큰을 포함한 프롬프트 토큰화 함수"""
                prompt_chunks = prompt.split(DEFAULT_IMAGE_TOKEN)
                
                tokens = []
                for i, chunk in enumerate(prompt_chunks):
                    if i > 0:
                        tokens.append(image_token_index)
                    tokens.extend(tokenizer(chunk).input_ids)
                
                if return_tensors:
                    if return_tensors == 'pt':
                        return torch.tensor(tokens)
                    else:
                        raise ValueError(f"지원되지 않는 return_tensors 값: {return_tensors}")
                return tokens
        
        # 모델 로드
        tokenizer, model, image_processor, context_len = self._load_pretrained_model(state["model_path"])
        
        # 학습 모드 설정
        model.train()
        model.gradient_checkpointing_enable()
        
        # 데이터셋 로드
        vqa_rad_dataset_train = VQARAD(root_dir=state["target_dir"], split="train")
        vqa_rad_dataset_test = VQARAD(root_dir=state["target_dir"], split="test")
        
        # 대화 템플릿 설정
        conv = conv_templates["mistral_instruct"]
        
        # DataCollator 설정
        collate_fn = DataCollator(
            tokenizer=tokenizer,
            split="train",
            conversation_template=conv,
            pad_token_id=tokenizer.pad_token_id,
            image_processor=image_processor
        )
        
        # 메모리 사용량 표시
        print("GPU 메모리 사용량:")
        for i in range(torch.cuda.device_count()):
            allocated = torch.cuda.memory_allocated(i) / (1024**3)
            reserved = torch.cuda.memory_reserved(i) / (1024**3)
            print(f"GPU {i}: {allocated:.2f} GB 할당, {reserved:.2f} GB 예약")
        
        # 상태 업데이트
        return {
            **state,
            "tokenizer": tokenizer,
            "model": model,
            "collate_fn": collate_fn,
            "vqa_rad_dataset_train": vqa_rad_dataset_train,
            "vqa_rad_dataset_test": vqa_rad_dataset_test,
            "context_len": context_len
        }
    
    def _load_pretrained_model(self, model_path, device_map="auto"):
        """모델 로드 함수"""
        # GPU 메모리 정리
        torch.cuda.empty_cache()
        gc.collect()
        torch.cuda.synchronize()
        
        print(f"모델 로드 중: {model_path}")
        
        # 양자화 설정
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        
        # 메모리 설정 (GPU당 사용 가능한 메모리 설정)
        gpu_count = torch.cuda.device_count()
        max_memory = {i: "10GB" for i in range(gpu_count)}
        max_memory["cpu"] = "24GB"
        
        # 모델 로드
        model = LlavaMistralForCausalLM.from_pretrained(
            model_path,
            quantization_config=quantization_config,
            device_map=device_map,
            max_memory=max_memory,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True
        )
        
        # 토크나이저 로드
        tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            use_fast=False,
            padding_side="right"
        )
        
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # 비전 타워 로드
        vision_tower = model.get_vision_tower()
        if not vision_tower.is_loaded:
            print("비전 타워 로드 중...")
            vision_tower.load_model()
            vision_tower.to(dtype=torch.bfloat16)
            print("비전 타워 로드 완료")
        
        # 패치 적용: 이미지 인코딩 메서드 동기화 (디바이스 일관성 유지)
        import types
        
        def synchronized_encode_images(self, images):
            """디바이스 동기화된 이미지 인코딩 메서드"""
            if images is None:
                return None
            
            # 필요한 모듈 가져오기
            vision_tower = self.get_model().get_vision_tower()
            mm_projector = self.get_model().mm_projector
            
            # 현재 디바이스 확인
            vision_device = next(vision_tower.parameters()).device
            mm_device = next(mm_projector.parameters()).device
            
            # 원래 이미지 디바이스 기억
            original_img_device = images.device
            
            # 1단계: vision tower와 이미지 디바이스 동기화
            if images.device != vision_device:
                images = images.to(vision_device)
            
            # 2단계: 이미지 특징 추출
            with torch.no_grad():
                image_forward_out = vision_tower.vision_tower(
                    images,
                    output_hidden_states=True
                )
                image_features = vision_tower.feature_select(image_forward_out)
            
            # 3단계: mm_projector와 이미지 특징 디바이스 동기화
            if image_features.device != mm_device:
                image_features = image_features.to(mm_device)
            
            # 4단계: 이미지 특징 프로젝션
            image_features = mm_projector(image_features)
            
            return image_features
        
        # 패치 적용
        model.encode_images = types.MethodType(synchronized_encode_images, model)
        
        # 로터리 임베딩 패치 (GPU 간 텐서 위치 불일치 해결)
        from transformers.models.mistral.modeling_mistral import apply_rotary_pos_emb, rotate_half
        
        # 패치된 rotate_half 함수 정의
        def patched_rotate_half(x):
            """GPU 동기화된 rotate_half 함수"""
            x1 = x[..., : x.shape[-1] // 2]
            x2 = x[..., x.shape[-1] // 2 :]
            return torch.cat((-x2, x1), dim=-1)
        
        # 패치된 apply_rotary 함수 정의
        def patched_apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
            """GPU 동기화된 로터리 포지션 임베딩 적용 함수"""
            # 기준 디바이스 확인
            q_device = q.device
            
            # 모든 텐서가 같은 디바이스에 있는지 확인하고 필요시 이동
            if cos.device != q_device:
                cos = cos.to(q_device)
            if sin.device != q_device:
                sin = sin.to(q_device)
            
            # 원본 함수 로직
            cos = cos.unsqueeze(unsqueeze_dim)
            sin = sin.unsqueeze(unsqueeze_dim)
            
            q_embed = (q * cos) + (patched_rotate_half(q) * sin)
            k_embed = (k * cos) + (patched_rotate_half(k) * sin)
            
            return q_embed, k_embed
        
        # 원본 함수 패치 적용
        import transformers.models.mistral.modeling_mistral as mistral_module
        mistral_module.rotate_half = patched_rotate_half
        mistral_module.apply_rotary_pos_emb = patched_apply_rotary_pos_emb
        
        print("로터리 임베딩 패치 완료!")
        
        # 컨텍스트 길이 계산
        if hasattr(model.config, "max_sequence_length"):
            context_len = model.config.max_sequence_length
        elif hasattr(model.config, "max_position_embeddings"):
            context_len = model.config.max_position_embeddings
        else:
            context_len = 2048
        
        return tokenizer, model, vision_tower.image_processor, context_len

In [7]:
class ModelTrainer:
    def __init__(self):
        pass
        
    def train(self, state: LLaVAMedPipelineState) -> LLaVAMedPipelineState:
        """모델 학습 및 저장"""
        print("===== 모델 학습 =====")
        
        # Wandb 초기화
        wandb.init(
            project="fnf-classification",
            name=f"fnf-classification-{datetime.now().strftime('%Y%m%d-%H%M')}",
            config={
                "model_name": state["model_path"],
                "learning_rate": 2e-5,
                "epochs": 5,
                "batch_size": 1,
                "gradient_accumulation_steps": 4,
                "lora_r": 8,
                "lora_alpha": 16
            }
        )
        
        # 양자화 모델을 학습 가능한 상태로 준비
        print("양자화 모델을 학습 가능한 상태로 준비 중...")
        model = prepare_model_for_kbit_training(state["model"])
        
        # LoRA 설정
        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=["k_proj", "q_proj", "v_proj", "out_proj"],
            bias="none",
            task_type="CAUSAL_LM"
        )
        
        # LoRA 모델 생성
        peft_model = get_peft_model(model, lora_config, "default")
        peft_model.config.use_cache = False
        peft_model.print_trainable_parameters()
        
        # 학습 설정
        training_args = TrainingArguments(
            output_dir=state["output_dir"],
            report_to="wandb",
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            logging_steps=5,
            learning_rate=2e-5,
            logging_strategy="epoch",
            save_strategy="epoch",
            num_train_epochs=5,
            warmup_ratio=0.03,
            weight_decay=0.01,
            remove_unused_columns=True,
            gradient_checkpointing=True,
            fp16=False,
            bf16=True,
            optim='paged_adamw_8bit',
            # DeepSpeed ZeRO-3 설정
            deepspeed={
                "zero_optimization": {
                    "stage": 3,
                    "overlap_comm": True,
                    "contiguous_gradients": True,
                    "reduce_bucket_size": "auto",
                    "stage3_prefetch_bucket_size": "auto",
                    "stage3_param_persistence_threshold": "auto"
                },
                "bf16": {
                    "enabled": True
                },
                "zero_allow_untested_optimizer": True
            }
        )
        
        # Trainer 초기화
        trainer = Trainer(
            model=peft_model,
            args=training_args,
            train_dataset=state["vqa_rad_dataset_train"],
            data_collator=state["collate_fn"]
        )
        
        # 학습 실행
        print("학습 시작...")
        trainer.train()
        print("학습 완료!")
        
        # 모델 저장
        lora_save_path = os.path.join(state["output_dir"], "lora_trained_model")
        trainer.save_model(lora_save_path)
        print(f"LoRA 모델 저장 완료: {lora_save_path}")
        
        # 모델 병합 및 저장
        try:
            # GPU 메모리 정리
            torch.cuda.empty_cache()
            gc.collect()
            
            # 기본 모델 로드
            print("기본 모델 로드 중...")
            base_model = AutoModelForCausalLM.from_pretrained(state["model_path"])
            
            # 학습된 모델 로드 및 병합
            print("학습된 모델 병합 중...")
            trained_model = PeftModel.from_pretrained(base_model, lora_save_path)
            merged_trained_model = trained_model.merge_and_unload()
            
            # 병합된 모델 저장
            merged_save_path = os.path.join(state["output_dir"], "merged_trained_model")
            merged_trained_model.save_pretrained(merged_save_path)
            state["tokenizer"].save_pretrained(merged_save_path)
            
            print(f"모델 병합 및 저장 완료: {merged_save_path}")
        
        except Exception as e:
            print(f"모델 병합 오류: {e}")
            import traceback
            traceback.print_exc()
        
        finally:
            # 메모리 정리
            torch.cuda.empty_cache()
            gc.collect()
            
            if wandb.run is not None:
                wandb.finish()
        
        return {**state, "training_completed": True}

In [8]:
class LLaVAMedPipeline:
    def __init__(self, source_dir, target_dir, model_path, output_dir, train_ratio=0.8, seed=42):
        self.source_dir = source_dir
        self.target_dir = target_dir
        self.model_path = model_path
        self.output_dir = output_dir
        self.train_ratio = train_ratio
        self.seed = seed
        
        # 초기 상태 설정
        self.initial_state = LLaVAMedPipelineState(
            source_dir=source_dir,
            target_dir=target_dir,
            model_path=model_path,
            output_dir=output_dir,
            train_ratio=train_ratio,
            seed=seed,
            train_dir="",
            test_dir="",
            processed_items=0,
            tokenizer=None,
            model=None,
            collate_fn=None,
            vqa_rad_dataset_train=None,
            vqa_rad_dataset_test=None,
            context_len=0,
            training_completed=False,
            lora_save_path="",
            merged_save_path=""
        )
        
        # 노드 클래스 인스턴스 생성
        self.data_processor = DataProcessor()
        self.dataset_preparation = DatasetPreparation()
        self.model_trainer = ModelTrainer()
        
        # 파이프라인 그래프 구성
        self.workflow = self._create_workflow()
        
    def _create_workflow(self):
        # TypedDict 상태 유형을 사용하는 StateGraph 생성 (초기 상태는 invoke 시 전달)
        workflow = StateGraph(LLaVAMedPipelineState)
        
        # 노드 추가
        workflow.add_node("process_and_save", self.data_processor.process)
        workflow.add_node("prepare_dataset", self.dataset_preparation.prepare)
        workflow.add_node("train_model", self.model_trainer.train)
        
        # 엣지 추가
        workflow.add_edge("process_and_save", "prepare_dataset")
        workflow.add_edge("prepare_dataset", "train_model")
        workflow.add_edge("train_model", END)
        
        # 입력 설정
        workflow.set_entry_point("process_and_save")
        
        # 컴파일
        return workflow.compile()
        
    def run(self):
        """파이프라인 실행"""
        print("===== LLaVA-Med 학습 파이프라인 시작 =====")
        # 초기 상태를 전달
        result = self.workflow.invoke(self.initial_state)
        print("===== LLaVA-Med 학습 파이프라인 완료 =====")
        return result


In [ ]:
if __name__ == "__main__":
    # 경로 설정
    source_dir = '/mnt/nas_backup/고효진/FNF/dataset'
    target_dir = '/mnt/nas_backup/고효진/FNF/dataset/generated_answer/splitted_dataset'
    model_path = "microsoft/llava-med-v1.5-mistral-7b"
    output_dir = "/mnt/nas_backup/고효진/FNF/save_model/LLaVAMed_trained_model_generated_answer"
    
    # 파이프라인 생성 및 실행
    pipeline = LLaVAMedPipeline(
        source_dir=source_dir,
        target_dir=target_dir,
        model_path=model_path,
        output_dir=output_dir
    )
    
    # 파이프라인 실행
    result = pipeline.run()

LLaVA-Med 모델 초기화 중...
모델 로드 중: microsoft/llava-med-v1.5-mistral-7b


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.embeddings.class_embedding', 'model.vision_tower.vision_tower.vision_model.embeddings.patch_embedding.weight', 'model.vision_tower.vision_tower.vision_model.embeddings.position_embedding.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.la

비전 타워 로드 중...
비전 타워 로드 완료
===== LLaVA-Med 학습 파이프라인 시작 =====
===== 이미지 분석 및 VQARAD 형식으로 변환 =====
총 12704개의 메타데이터 파일 발견


  0%|                                                                                                                                                                                                                  | 0/10 [00:00<?, ?it/s]

...

===== 이미지 분석 완료: 6t000.dcm =====



  0%|                                                                                                                                                                                                                  | 0/10 [00:00<?, ?it/s]

이미지 파일 경로: /mnt/nas_backup/고효진/FNF/FNF_origin/FNF/Final_data/datasets/raw_internal/6/6a000.dcm
이미지 파일 경로: /mnt/nas_backup/고효진/FNF/FNF_origin/FNF/Final_data/datasets/raw_internal/6/6a000.dcm
이미지 파일 크기: 8510.21 KB
DICOM 이미지 크기: (2009, 2017)
DICOM 이미지 타입: 1.2.840.10008.5.1.4.1.1.1.1
질문 글자 수: 933
질문 내용: 
                        [Instructions]
                        You are an AI assistant specialized ...

----- 모델 인자 설정 -----
모델 경로: microsoft/llava-med-v1.5-mistral-7b
대화 모드: mistral_instruct
최대 토큰 수: 500
온도(temperature): 0.7

----- 모델 실행 시작 -----


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at microsoft/llava-med-v1.5-mistral-7b were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.embeddings.class_embedding', 'model.vision_tower.vision_tower.vision_model.embeddings.patch_embedding.weight', 'model.vision_tower.vision_tower.vision_model.embeddings.position_embedding.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.layer_norm2.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc1.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.0.mlp.fc2.bias', 'model.vision_tower.vision_tower.vision_model.encoder.la